#LogisticRegression

In [0]:
CATALOG_NAME = "ml_training_dev"
SCHEMA_NAME = "gold"
TABLE_NAME = "customers_orders_category"

In [0]:
df_feature_table = spark.table(f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}").drop("loadDate")

In [0]:
df_sample = df_feature_table.sample(fraction=0.1, seed=42)
df_train, df_test = df_sample.randomSplit([0.8, 0.2], seed=42)
display(df_train)

In [0]:
display(df_test)

In [0]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import LogisticRegression

# Define binary target column based on 'price'
threshold = 1000
df_train = df_train.withColumn("is_high_spender", (col("price") > threshold).cast("integer"))
df_test = df_test.withColumn("is_high_spender", (col("price") > threshold).cast("integer"))

In [0]:
# Index categorical columns
indexers = [
    StringIndexer(inputCol="product_category_name", outputCol="product_category_name_index"),
    StringIndexer(inputCol="customer_unique_id", outputCol="customer_unique_id_index")
]
for indexer in indexers:
    model = indexer.fit(df_train)
    df_train = model.transform(df_train)
    df_test = model.transform(df_test)
    del model

In [0]:
# Assemble features
feature_cols = ["product_category_name_index", "customer_unique_id_index", "price"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_train = assembler.transform(df_train)
df_test = assembler.transform(df_test)

# Fit logistic regression model
lr = LogisticRegression(featuresCol="features", labelCol="is_high_spender")
model = lr.fit(df_train)

# Display feature weights (coefficients)
display(model.coefficients)

In [0]:
for indexer in indexers:
    model = indexer.fit(df_lr); df_lr = model.transform(df_lr); del model
# Após esse passo, df_lr contém novas colunas indexadas, mas display falha se você tentar exibir apenas model.coefficients, pois model.coefficients não é um DataFrame.

# Select features for logistic regression
feature_cols = ["product_category_name_index", "customer_unique_id_index", "price"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_lr = assembler.transform(df_lr)

In [0]:
# Fit logistic regression model
lr = LogisticRegression(featuresCol="features", labelCol="is_high_spender")
model = lr.fit(df_lr)

# Display feature weights (coefficients)
display(model.coefficients)